<img src="../logo.png" alt="Vegeta" width="240">

# STERNA — a 1.5 m electric survey fixed wing for coastal work (51)

Coastal surveys, environmental monitoring and maritime search support want a small aircraft that one person carries
to the shore, throws, and lands in a field: **STERNA** is a 1.5 m shoulder-wing tractor with a folding propeller, a
payload pallet in the belly just ahead of the CG, plug-in wing panels on one straight joiner tube and a low
conventional tail. After the tern (a slender body, long lightly loaded outer panels) and the A6M Zero (mass
discipline, big control surfaces).

This notebook is the design **in simplified form, each step on one of Vegeta's tools**:

1. **Design** — the airframe in CAD (Dedalus), its measurements.
2. **Section** — the wing section's polar (Zephyr).
3. **Mass and balance** — the budget, the loading cases, the CG.
4. **Calculations** — the lattice (Ogma), the neutral point, speeds; the drive (Boreas).
5. **CFD** — the whole airframe at four angles on a screening mesh (Aeromant / OpenFOAM), and the model fitted to it.
6. **FEA** — the wing joiner tube at 6 g (Talos / CalculiX) against the hand value.
7. **Performance** — cruise power, and what one charge gives *in this model*.
8. **Flight** — one mission in MuJoCo (hand launch, climb, a leg, an orbit, a power-off glide, a belly landing) and
   its video.

**What "simplified" means here.** Four CFD cases on the coarsest preset, fully turbulent, no propeller. One FEA part
with an isotropic stand-in material. A NACA 4-digit stand-in for the low-Reynolds section. Hand autopilot gains.
Hand-estimated lateral derivatives, inertia and component masses. No wind limit, link range or detection performance
is derived anywhere, and the endurance in section 7 is a model output with a stated band, not a specification.

In [ ]:
# DEBUG — OFF here: the point of this notebook is the real (simplified) solver runs. True mocks them (vegeta.mock).
DEBUG = False
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)        # the lab's folders on sys.path, the mocks when on

In [ ]:
import json, math, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import display, Image, Video
from vegeta import dedalus, zephyr, talos, aeromant, cache
from vegeta.dedalus import viz as dviz
from vegeta.aeromant import viz as aviz
from vegeta.talos import viz as tviz
import arigo1_model as am
import sterna as st
import sterna_mass as sm
import sterna_model as mo
import sterna_cfd as sc

RUNS = Path("../_runs/sterna"); RUNS.mkdir(parents=True, exist_ok=True)       # solver cases (gitignored)
OUT = Path("output/51_sterna"); OUT.mkdir(parents=True, exist_ok=True)        # pictures and the video (gitignored)
DATA = Path("../designs/data")                                                 # what is kept: the CFD rows, the record
cache.notebook("51_sterna")
os.environ.setdefault("MUJOCO_GL", "egl")
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
INK, ACC, TEAL = "#13232C", "#C8420D", "#0C6F68"

In [ ]:
import yanus                                       # Ozaru's requirement checks: what this notebook needs on this machine
from yanus import Package
NEEDS = [Package("vegeta.dedalus"), Package("vegeta.zephyr"), Package("vegeta.talos"), Package("vegeta.aeromant"), Package("vegeta.boreas"), Package("mujoco")]
print(yanus.table(yanus.check_needs(NEEDS)))

## 1. Design — the airframe in CAD

`sterna.Sterna` is a Dedalus design. The fuselage is not a body of revolution: its top line, bottom line and
half-width are tables over the length and the sections are superellipses between them (a flat-sided, round-cornered
box 100 × 125 mm at the wing). The wing is flat over the centre 500 mm and cranked up 5° outboard, so one straight tube
joins the two panels. Frame: x aft from the spinner tip, y right, z up from the thrust line; mm.

In [ ]:
p = st.resolve()
L = st.Sterna.layout(p)
geom = st.Sterna().generate()
meas = geom.measure()
(DATA / "sterna").mkdir(exist_ok=True)
geom.export_step(str(RUNS / "sterna.step"))
print(geom, "| valid:", meas["valid"], "| solids:", meas["n_solids"], "| wetted area %.3f m2" % (meas["surface_area"] / 1e6))
pd.DataFrame([
    ("span", p["span"], "mm"), ("length", L["length"], "mm"), ("height on the rails", L["height"], "mm"),
    ("wing area", L["S_mm2"] / 1e6, "m2"), ("aspect ratio", L["AR"], ""), ("mean aerodynamic chord", L["mac"], "mm"),
    ("MAC leading edge station", L["x_mac_le"], "mm"), ("design CG station (30 % MAC class)", p["cg_x"], "mm"),
    ("tailplane area", L["S_h_mm2"] / 1e6, "m2"), ("tail arm", L["l_h"], "mm"), ("tail volume V_h", L["V_h"], ""),
    ("fin area", L["S_v_mm2"] / 1e6, "m2"), ("fin volume V_v", L["V_v"], ""),
    ("propeller tip below the rails", L["prop_below_rails"], "mm"),
], columns=["quantity", "value", "unit"]).round(3)

In [ ]:
import pyvista as pv
pv.OFF_SCREEN = True
parts = st.Sterna().parts(p)
pl = pv.Plotter(off_screen=True, window_size=(1400, 800))
for name, colour in (("fuselage", "#E9EDEF"), ("wing", "#F6F7F8"), ("tail", "#F6F7F8"), ("props", "#C8420D")):
    g = dedalus.Geometry.from_cadquery(parts[name], name=name)
    f = RUNS / f"view_{name}.stl"; g.export_stl(str(f), tolerance=0.3)
    pl.add_mesh(pv.read(str(f)), color=colour, opacity=0.35 if name == "props" else 1.0, smooth_shading=True, split_sharp_edges=True)
pl.set_background("white"); pl.enable_lightkit()
pl.camera_position = [(-1500.0, -1700.0, 1000.0), (520.0, 0.0, 20.0), (0.0, 0.0, 1.0)]
pl.camera.zoom(1.35)
pl.screenshot(str(DATA / "sterna_3d.png")); pl.close()
Image(str(DATA / "sterna_3d.png"), width=900)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, normal, pos, title in ((axes[0], "y", 0.0, "centre plane (y = 0)"), (axes[1], "x", 402.0, "at the joiner (x = 402)"), (axes[2], "z", 38.0, "wing plane (z = 38)")):
    for poly in dviz.section_polylines(geom, normal, pos):
        a = np.asarray(poly)
        ax.plot(a[:, 0], a[:, 1], color=INK, lw=1.0)
    ax.set_aspect("equal"); ax.set_title(title, fontsize=10)
plt.tight_layout(); plt.show()

## 2. Section — the wing's polar

The design calls for a low-Reynolds section of the SD7062 (root) / SD7037 (tip) class. Its coordinates are not in the
repository, so the CAD and this polar use a **NACA 3412 stand-in** (3 % camber at 40 %, 12 % thick): close in camber
and thickness, not the same section. Zephyr's polar at the mean chord's Reynolds number at 14 m/s gives the 2-D
maximum lift the stall speed rests on.

In [ ]:
Re = 14.0 * L["mac"] / 1e3 / 1.46e-5
section = zephyr.naca("3412")
polar = zephyr.analyze(section, alphas=range(-4, 19), Re=Re)
ps = polar.summary(cl_design=0.55)
CL_MAX_2D = ps["cl_max"]
print(f"Re {Re:,.0f} | source {ps['source']} | cl_max {ps['cl_max']:.2f} at {ps['alpha_cl_max_deg']:.0f} deg | alpha0 {ps['alpha0_deg']:.1f} deg | "
      f"cd_min {ps['cd_min']:.4f} | cm0 {ps['cm0']:.3f} | cl 4 deg past the stall / cl_max {ps['cl_4deg_past_stall_ratio']:.2f}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(polar.alpha_deg, polar.cl, color=INK); ax[0].axhline(CL_MAX_2D, color=ACC, lw=0.8, ls="--"); ax[0].set_xlabel("alpha [deg]"); ax[0].set_ylabel("cl")
ax[1].plot(polar.cd, polar.cl, color=INK); ax[1].set_xlabel("cd"); ax[1].set_ylabel("cl"); ax[1].set_xlim(0, 0.06)
plt.tight_layout(); plt.show()
print(f"wing CL_max = 0.9 x {CL_MAX_2D:.2f} = {0.9 * CL_MAX_2D:.2f} (Raymer's unswept-wing factor). The concept report assumed 1.1: on the cautious side of this.")

## 3. Mass and balance

Every item with its mass, its station and where the number comes from (`sterna_mass`). Nothing is weighed: the
structure is an areal or linear density times the CAD's areas and lengths, the equipment is class values with margins.
The payload pallets (0–250 g at station 250) and the battery modules (one or two 4S1P, about 310 g each) are
**alternatives**; the battery tray slides between stations 320 and 495 under the joiner and each case has its tray
position.

In [ ]:
b0 = sm.budget(p, L, "daylight_1")
items = pd.DataFrame(b0["items"])
display(items.groupby("group", sort=False)["g"].sum().round(0).to_frame("grams").T)
display(items)
print(f"empty (no pallet, no battery): {b0['empty_g']:.0f} g at station {b0['empty_x_mm']:.0f} mm | one module: {sm.MODULE_G} g, {sm.MODULE_WH:.1f} Wh nominal")

In [ ]:
rows = []
for name in sm.CASES:
    b = sm.budget(p, L, name)
    ideal = sm.tray_for(p, L, b["pallet"]["name"], b["battery"]["modules"])
    half = b["battery"]["modules"] * sm.MODULE_LEN / 2
    rows.append({"case": name, "pallet g": b["pallet"]["g"], "modules": b["battery"]["modules"], "tray centre": b["battery"]["x_mm"],
                 "tray for 30 %": round(ideal), "fits the bay": st.BATTERY_BAY[0] + half <= b["battery"]["x_mm"] <= st.BATTERY_BAY[1] - half + 0.5,
                 "take-off kg": b["total_g"] / 1000, "CG station": b["x_cg_mm"], "CG % MAC": 100 * b["cg_mac"],
                 "kg/m2": b["total_g"] / 1000 / (L["S_mm2"] / 1e6), "within 2.45 kg": b["within_mtow"]})
cases = pd.DataFrame(rows).round(2)
display(cases)
fig, ax = plt.subplots(figsize=(8.5, 3.0))
ax.axvspan(26, 34, color=TEAL, alpha=0.12, label="provisional range 26-34 % MAC")
ax.scatter(cases["CG % MAC"], range(len(cases)), color=ACC, zorder=3)
ax.set_yticks(range(len(cases))); ax.set_yticklabels(cases["case"]); ax.invert_yaxis(); ax.set_xlabel("CG [% MAC]"); ax.set_xlim(20, 40); ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

## 4. Calculations — lattice, stability, speeds, the drive

`sterna_model.coefficients` builds the coefficient set the lab's flight model takes (`arigo1_model.aero`): the lift
curve, the neutral point and the elevator from the vortex lattice (`ogma.aero.vlm` on the wing with its crank and
washout, and the tailplane), the rest by hand formulas marked as such. The lattice has no fuselage and no propeller:
4 % MAC is taken off its neutral point for them, a hand value that section 5 replaces.

In [ ]:
C0 = mo.coefficients("daylight_1", cl_max_2d=CL_MAX_2D)
np_pct = lambda x_m: 100 * (x_m * 1e3 - L["x_mac_le"]) / L["mac"]
print(f"lattice: CL0 {C0['CL0']:.3f}, CLa {C0['CLa']:.2f} /rad, neutral point {C0['x_np_lattice_m'] * 1e3:.0f} mm = {np_pct(C0['x_np_lattice_m']):.0f} % MAC")
print(f"with the fuselage allowance: {np_pct(C0['x_np_m']):.0f} % MAC -> static margin {100 * C0['static_margin']:.0f} % MAC at the CG of {np_pct(C0['x_cg_m']):.0f} %")
print(f"(the concept report's tail-volume estimate was about 42 % MAC: the lattice is more stable than the hand sum; the CFD decides)")
sp = pd.DataFrame([{"case": n, "mass kg": c["mass"], **{k: v for k, v in mo.speeds(c).items() if k.startswith("V_") or k == "LD_max"}}
                   for n in ("daylight_1", "dual_1", "daylight_2", "limit_2") for c in [mo.coefficients(n, cl_max_2d=CL_MAX_2D)]]).round(2)
display(sp)
print("stall with the report's CL_max of 1.1 instead:", ", ".join(f"{math.sqrt(m * am.G / (0.5 * am.RHO * C0['S'] * 1.1)):.1f}" for m in sp["mass kg"]), "m/s")

In [ ]:
drive = mo.propulsion(modules=1, kv=700.0)
tt = mo.thrust_table(drive)
W_max = sm.MTOW_G / 1000 * am.G
print(f"{drive.motor.name}; static thrust {tt['T0_N']:.1f} N = {tt['T0_N'] / W_max:.2f} x weight at 2.45 kg (requirement 0.70), {tt['table_A'][0]:.0f} A; current limited: {tt['current_limited']}")
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(tt["table_V"], tt["table_T"], color=INK, label="full-throttle thrust (Boreas)")
V = np.linspace(9, 28, 60)
for name, col in (("daylight_1", TEAL), ("limit_2", ACC)):
    c = mo.coefficients(name, cl_max_2d=CL_MAX_2D)
    ax.plot(V, [mo.power_required(c, v) / v for v in V], color=col, ls="--", label=f"drag, {name} (hand CD0)")
ax.set_xlabel("airspeed [m/s]"); ax.set_ylabel("N"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
C0["thrust"] = tt

## 5. CFD — the whole airframe at four angles

Aeromant's external RANS template on the `screening` preset: the airframe without the propeller at −2°, 1°, 4° and 8° and
14 m/s, moments about the design CG. The model is fitted to the first three; the 8° case shows where the lift curve
starts to bend on this mesh. About a minute per case on four cores. The rows are kept in
`designs/data/sterna_cfd_screening.yaml`; on a machine without OpenFOAM the notebook reads them from there.

What to take from it: the lift slope, the neutral point with the fuselage in it, and an upper estimate of the drag
(fully turbulent, wall functions, a coarse mesh, at a Reynolds number where the real boundary layers are partly
laminar). The stall is not in these cases.

In [ ]:
CFD_FILE = DATA / "sterna_cfd_screening.yaml"
try:
    env = sc.environment(); HAVE_FOAM = True
    print("OpenFOAM:", env.flavor(), env.version())
except Exception as exc:
    HAVE_FOAM = False; print("no OpenFOAM here:", exc)
cfd_cases = {}
if HAVE_FOAM and not DEBUG:
    cfd_rows = []
    for al in sc.ALPHAS:
        cfd_cases[al] = sc.case(RUNS / "cfd", alpha_deg=al, env=env)
        res = cfd_cases[al].ensure(run=True, processors=4, timeout=3600, cache=f"cfd_a{al:+.0f}")
        cfd_rows.append(sc.row(al, res))
    CFD_FILE.write_text(yaml.safe_dump({"aircraft": "STERNA", "speed_m_s": sc.SPEED, "quality": "screening", "template": "rans_ksst_external",
                                        "note": "airframe without propeller; Cm about the design CG on the mean chord", "rows": cfd_rows}, sort_keys=False))
else:
    cfd_rows = yaml.safe_load(CFD_FILE.read_text())["rows"]
    print("rows read from", CFD_FILE)
cfd = pd.DataFrame(cfd_rows)
cfd["Cl lattice"] = [C0["CL0"] + C0["CLa"] * math.radians(a) for a in cfd["alpha_deg"]]
cfd.round(4)

In [ ]:
fit_rows = [r for r in cfd_rows if r["alpha_deg"] <= sc.FIT_MAX_ALPHA]
C = mo.calibrate(C0, fit_rows)
last = cfd_rows[-1]
print(f"at {last['alpha_deg']:.0f} deg: CL {last['Cl']:.2f} against {C['CL0'] + C['CLa'] * math.radians(last['alpha_deg']):.2f} on the fitted line, Cm {last['Cm_cg']:+.3f} against {C['Cm0'] + C['Cma'] * math.radians(last['alpha_deg']):+.3f}: the lift bends over and the nose-down moment grows (separation on a coarse, fully turbulent mesh: an early, unreliable picture of the stall)")
print(f"lift slope: CFD {C['CLa']:.2f} /rad, lattice {C0['CLa']:.2f} | CL0: CFD {C['CL0']:.3f}, lattice {C0['CL0']:.3f}")
print(f"neutral point: CFD {np_pct(C['x_np_m']):.0f} % MAC, lattice with allowance {np_pct(C0['x_np_m']):.0f} %, lattice alone {np_pct(C0['x_np_lattice_m']):.0f} % -> static margin {100 * C['static_margin']:.0f} % MAC")
print(f"polar fit: CD0 {C['CD0']:.4f}, k {C['k']:.4f} (hand: {C0['CD0']:.4f}, {C0['k']:.4f}) -> L/D max {mo.speeds(C)['LD_max']:.1f} (hand polar {mo.speeds(C0)['LD_max']:.1f})")
fig, ax = plt.subplots(1, 3, figsize=(12, 3.3))
a = np.linspace(-3, 9, 30)
ax[0].plot(a, C0["CL0"] + C0["CLa"] * np.radians(a), color=INK, ls="--", label="lattice"); ax[0].scatter(cfd["alpha_deg"], cfd["Cl"], color=ACC, zorder=3, label="CFD"); ax[0].set_xlabel("alpha [deg]"); ax[0].set_ylabel("CL"); ax[0].legend(fontsize=8)
cl = np.linspace(0, 1.1, 40)
ax[1].plot(C0["CD0"] + C0["k"] * cl ** 2, cl, color=INK, ls="--", label="hand polar"); ax[1].plot(C["CD0"] + C["k"] * cl ** 2, cl, color=ACC, lw=0.8, label="CFD fit"); ax[1].scatter(cfd["Cd"], cfd["Cl"], color=ACC, zorder=3); ax[1].set_xlabel("CD"); ax[1].set_ylabel("CL"); ax[1].legend(fontsize=8)
ax[2].scatter(cfd["alpha_deg"], cfd["Cm_cg"], color=ACC, zorder=3, label="CFD"); ax[2].plot(a, C["Cm0"] + C["Cma"] * np.radians(a), color=ACC, lw=0.8); ax[2].plot(a, C0["Cm0"] + C0["Cma"] * np.radians(a), color=INK, ls="--", label="lattice + allowance"); ax[2].set_xlabel("alpha [deg]"); ax[2].set_ylabel("Cm about the CG"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
if cfd_cases and Path(cfd_cases[4.0].workdir).exists():
    fig = aviz.plot_section(cfd_cases[4.0], "U", zoom=2)
    plt.gcf().set_size_inches(10, 4.2); plt.xlim(-0.4, 1.9); plt.ylim(-0.55, 0.55); plt.title("speed on the centre plane, alpha 4 deg, 14 m/s", fontsize=10); plt.show()
else:
    print("the flow picture needs the case directory (the cache keeps only the coefficients)")

## 6. FEA — the wing joiner at 6 g

The two wing panels sit on one carbon tube, 14 mm outside diameter with a 1 mm wall. At 6 g and the 2.45 kg mass limit
each half-wing carries half the load at about 43 % of the half-span, 322 mm from the centre line and 272 mm from the
fuselage wall, where the sleeve reacts it: a bending moment of about 20 N m in the tube (the concept report took the
lever from the centre line, 23 N m: the upper bound). The part is the tube from the wall (fixed) to its end in the panel, 140 mm, with an
end load that gives that moment at the wall. Talos meshes it with second-order tetrahedra and CalculiX solves it.

Simplified: the tube's lay-up is not modelled (an isotropic stand-in with an axial modulus; stress in a statically
determinate beam does not depend on it), the panel's socket spreads the load in reality, and 600 MPa is a hand
allowable for a wrapped carbon tube in bending, not a tested one.

In [ ]:
n_limit, y_lift = 6.0, 0.43 * 750.0
M_wall = n_limit * sm.MTOW_G / 1000 * am.G / 2 * (y_lift - 50.0) / 1000                 # N m at the fuselage wall (y = 50 mm)
J = st.JOINER
F_end = M_wall * 1000 / J["in_panel"]
Z = math.pi * (J["od"] ** 4 - (J["od"] - 2 * J["wall"]) ** 4) / (32 * J["od"])
sigma_hand = M_wall * 1000 / Z
st.Sterna().generate(part="joiner").export_step(str(RUNS / "joiner.step"))
print(talos.inspect_step(str(RUNS / "joiner.step"), units="mm-N-MPa").table())
cfrp = talos.Material("carbon tube, isotropic stand-in", youngs_modulus=110000, poissons_ratio=0.3, density=1.55e-9, yield_strength=600, source="HAND")
joiner = talos.StructuralModel(geometry=str(RUNS / "joiner.step"), units="mm-N-MPa", material=cfrp,
                               regions=[talos.SurfacesOnPlane("wall", "y", 50.0), talos.SurfacesOnPlane("end", "y", 50.0 + J["in_panel"])],
                               supports=[talos.FixedSupport("wall")], loads=[talos.Force("end", fz=F_end)],
                               mesh_settings=talos.MeshSettings(element_size=2.0, order=2))
mesh = joiner.mesh(str(RUNS / "joiner"))
fea = joiner.solve(str(RUNS / "joiner"))
m = fea.metrics
print(f"moment at the wall {M_wall:.1f} N m, end load {F_end:.0f} N | mesh {mesh.metrics['n_nodes']} nodes, {mesh.metrics['n_elements']} C3D10")
print(f"max von Mises {m['max_von_mises']:.0f} MPa at y = {m['max_von_mises_location'][1]:.0f} mm | hand M/Z {sigma_hand:.0f} MPa ({100 * (m['max_von_mises'] / sigma_hand - 1):+.0f} %)")
print(f"end deflection {m['max_displacement']:.2f} mm | reserve on the 600 MPa hand allowable {m['safety_factor_yield']:.1f} | reaction {m['reaction_total'][2]:.0f} N")

In [ ]:
plr = tviz.plot_results(fea, field="von_mises")
plr.window_size = (1100, 420); plr.set_background("white")
plr.screenshot(str(OUT / "joiner_von_mises.png")); plr.close()
Image(str(OUT / "joiner_von_mises.png"), width=800)

## 7. Performance — cruise power, and one charge in this model

Level flight on the CFD-fitted polar, the drive's operating point from Boreas for the drag at each speed, 12 W for
the avionics (hand), 80 % of the nominal energy used. **These minutes are a model output.** The screening CFD
over-states the drag, the propeller map is a generic section, the masses are estimates: the band below is the same
calculation on the hand polar. The number to trust is the one measured in flight.

In [ ]:
rows = []
for name in ("daylight_1", "dual_1", "daylight_2", "dual_2"):
    c_hand = mo.coefficients(name, cl_max_2d=CL_MAX_2D)
    c_cfd = mo.calibrate(c_hand, fit_rows)
    sysn = mo.propulsion(modules=c_hand["mass_budget"]["battery"]["modules"])
    for V in (12.0, 14.0, 16.0):
        a_, b_ = mo.cruise_point(c_cfd, sysn, V), mo.cruise_point(c_hand, sysn, V)
        rows.append({"case": name, "kg": c_cfd["mass"], "V m/s": V, "CL": a_["CL"], "L/D (CFD fit)": a_["L_D"], "L/D (hand)": b_["L_D"], "throttle": a_["throttle"],
                     "total W": a_["total_W"], "usable Wh": a_["usable_Wh"], "min (CFD fit)": a_["minutes"], "min (hand polar)": b_["minutes"], "km (CFD fit)": a_["minutes"] * 60 * V / 1000})
perf = pd.DataFrame(rows).round(2)
perf

## 8. Flight — one mission in MuJoCo, and the video

MuJoCo integrates the rigid body and the belly's contact with the ground; the forces are the lab's flight model on
the coefficients above (CFD-fitted lift, drag and pitching moment, hand lateral derivatives, the Boreas thrust map).
The mission (`sterna_mujoco.mission`): released level at 8 m/s by hand at full throttle, a climb to 40 m, a straight
leg at 14 m/s, one orbit at 30° bank, then the propeller stopped, a glide at 1.35 × stall, a flare and a belly landing.
Still air. The autopilot's gains are the lab's hand gains. The video shows the propeller as two real blades turning at the real speed: the rpm the Boreas drive needs for the thrust and
airspeed of each moment. That is several turns per frame, so each frame is exposed with motion blur, as a camera would record
it. When the throttle closes the propeller stops and the blades fold back along the nose. Flaps are not in the model: the touchdown is at the glide speed, faster than a flapped,
held-off landing would be.

In [ ]:
import sterna_mujoco as smj
Cf = dict(C); Cf["thrust"] = tt
sim = smj.Sim(Cf)
hist, mission = smj.mission(sim)
print(json.dumps(mission, indent=1))
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].plot(hist["t"], hist["z"], color=INK); ax[0].set_xlabel("t [s]"); ax[0].set_ylabel("height [m]")
ax[1].plot(hist["t"], hist["V"], color=INK, label="airspeed"); ax[1].axhline(mission["stall_m_s"], color=ACC, ls="--", lw=0.8, label="stall"); ax[1].plot(hist["t"], 10 * hist["throttle"], color=TEAL, lw=0.8, label="throttle x 10"); ax[1].set_xlabel("t [s]"); ax[1].legend(fontsize=8)
ax[2].plot(hist["x"], hist["y"], color=INK); ax[2].set_aspect("equal"); ax[2].set_xlabel("x [m]"); ax[2].set_ylabel("y [m]"); ax[2].set_title("ground track", fontsize=10)
for t_, name in mission["phases"]:
    ax[0].axvline(t_, color=ACC, lw=0.5); ax[0].text(t_, ax[0].get_ylim()[1], " " + name, fontsize=7, va="top", rotation=90)
plt.tight_layout(); plt.show()

In [ ]:
rows = []
for name, release in (("daylight_1", 8.0), ("daylight_1", 6.0), ("limit_2", 8.0), ("limit_2", 10.7)):
    c = mo.calibrate(mo.coefficients(name, cl_max_2d=CL_MAX_2D), fit_rows); c["thrust"] = mo.thrust_table(mo.propulsion(modules=c["mass_budget"]["battery"]["modules"]))
    s_ = smj.Sim(c); s_.start(max(release, 1.15 * mo.stall_speed(c)), smj.RELEASE_HEIGHT_M)
    h_, m_ = smj.mission(s_, release=release)
    rows.append({"case": name, "kg": m_["mass_kg"], "stall m/s": m_["stall_m_s"], "release m/s": release, "lowest point m": m_["launch_min_height_m"],
                 "safe after s": m_["time_to_safe_s"], "launch": "ok" if m_["time_to_safe_s"] else "ground contact"})
print("launches: released level at 1.9 m, full throttle, still air (10.7 m/s stands for the bungee line at the mass limit).")
print("A lowest point under about 1 m leaves no margin for a poor throw or a lull: those cases are the line's.")
pd.DataFrame(rows)

In [ ]:
mesh_file = smj.export_mesh(RUNS / "mesh")
blade_file = smj.export_blade(RUNS / "mesh")                 # one blade of the 11x7 folding propeller (dedalus.examples.Propeller)
sim = smj.Sim(Cf, mesh=mesh_file, blade=blade_file)
VIDEO = OUT / "sterna_mission.mp4"
rec = smj.Recorder(sim, VIDEO)
_, mission_v = smj.mission(sim, rec)
rec.close()
print(f"{rec.frames} frames, {rec.frames / rec.fps:.0f} s -> {VIDEO} ({VIDEO.stat().st_size / 1e6:.1f} MB)")
rpm = np.array(rec.rpm); ph = np.array(hist["phase"])[::max(1, len(hist["phase"]) // len(rpm))][:len(rpm)]
print("propeller rpm in the video: " + ", ".join(f"{name} {rpm[:len(ph)][ph == name].mean():.0f}" for name in ("launch", "climb", "orbit", "glide") if (ph == name).any())
      + f" | zero-thrust speed at {smj.CRUISE_M_S:.0f} m/s: {smj.CRUISE_M_S / (7 * 0.0254) * 60:.0f}")
Video(str(VIDEO), embed=False, width=800)

## 9. The record, and what this notebook does not settle

In [ ]:
record = {"aircraft": "STERNA", "notebook": "51_sterna",
          "geometry": {k: round(float(v), 4) for k, v in L.items() if k != "stations"},
          "section": {"stand_in": "NACA 3412", "Re": round(Re), "cl_max_2d": round(CL_MAX_2D, 3), "source": ps["source"]},
          "mass_cases": cases.to_dict("records"),
          "model": {k: (round(float(v), 5) if isinstance(v, (int, float)) else v) for k, v in C.items() if k in ("mass", "S", "c", "b", "CL0", "CLa", "CL_MAX", "CD0", "k", "Cm0", "Cma", "x_cg_m", "x_np_m", "static_margin", "calibrated_from")},
          "drive": {"motor": drive.motor.name, "static_thrust_N": round(tt["T0_N"], 2), "static_current_A": round(tt["table_A"][0], 1)},
          "cfd": cfd_rows,
          "fea_joiner": {"moment_Nm": round(M_wall, 2), "max_von_mises_MPa": round(m["max_von_mises"], 1), "hand_MPa": round(sigma_hand, 1), "end_deflection_mm": round(m["max_displacement"], 3)},
          "performance_model": perf.to_dict("records"), "mission": mission}
(DATA / "sterna_record.json").write_text(json.dumps(record, indent=1, default=str))
print("written:", DATA / "sterna_record.json")

**Not settled here** — each needs its own run or a test:

- the **stall**: its angle, how gentle the break is, and the real CL_max (a CFD sweep to 20° on a finer mesh, then flight);
- the **real section** (SD7062 / SD7037 coordinates through Zephyr, with a transition-aware backend);
- the **propeller on**: slipstream over the tail and the pallet, the installed thrust (Aeromant's rotor-disk cases);
- **sideslip and the lateral derivatives** (hand values here), gust response, and from them any **wind limit**;
- the **wing panel** itself in bending and torsion, the root fittings, the tailplane mounts, the landing loads on the rails;
- **every mass**: weigh the parts against the budget, then weigh and balance the aircraft;
- **endurance and range**: measured cruise power per loading case;
- **link range, antenna interference and detection performance**: none of it is modelled.